# TopicBank: Bank Creation Experiment

Here we are going to collect interpretable topics (automatically, using topic coherence) from multiple model training.
These topics constitute *topic bank*.
And then the topic bank is going to be used for estimating topic models quality in the notebook [TopicBank-Experiment: Model Validation](TopicBank-Experiment-ModelValidation.ipynb).

The process is repeated for several datasets (some of them are already downloadable using [TopicNet](https://github.com/machine-intelligence-laboratory/TopicNet) library).

# Contents<a id="contents"></a>

* [Data](#data)
    * [Coocs](#coocs)
        * [Lower Memory Consumption (or a Bit of Shamanism. Part 1)](#optimizing-memory)
    * [Documents for Coherence Scores](#docs-for-cohs)
        * [Lower Time Consumption in Case of Big Datasets (or a Bit of Shamanism. Part 2)](#optimizing-time)
* [Experiment](#experiment)
    * [Scores](#scores)
    * [Bank Creation](#bank-creation)
* [Postprocessing](#postprocessing)

In [1]:
# General imports

import dill
import itertools
import json
import numpy as np
import os
import pandas as pd
import sys

from enum import Enum
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from tqdm import tqdm
from typing import (
    Dict,
    Iterable,
)

%matplotlib inline

In [2]:
# Making `topnum` module visible for Python

sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
# Optimal number of topics

from topicnet.cooking_machine import Dataset

from topnum.data.vowpal_wabbit_text_collection import VowpalWabbitTextCollection
from topnum.scores import (
    PerplexityScore,
    SparsityPhiScore,
    SparsityThetaScore,
)
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.scores._base_coherence_score import (
    SpecificityEstimationMethod,
    TextType,
    WordTopicRelatednessType,
)
from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)
from topnum.search_methods import TopicBankMethod
from topnum.search_methods.topic_bank.topic_bank import TopicBank
from topnum.search_methods.topic_bank.one_model_train_funcs import (
    default_train_func,

    # Functions below are not used (but could have been)

#     regularization_train_func,
#     specific_initial_phi_train_func,
#     background_topics_train_func,

)

## Data<a id="data"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Loading data from disk, creating batches, dictionary, gathering cooccurrence statistics...

In [4]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [5]:
sorted(os.listdir(DATA_FOLDER_PATH))

['20_Newsgroups_NOOW.csv',
 '20_Newsgroups_NOOW__internals',
 '20_Newsgroups__internals',
 'MKB_10_NOOW.csv',
 'MKB_10_NOOW__internals',
 'MKB_10__internals',
 'Post_Science_NOOW.csv',
 'Post_Science_NOOW__internals',
 'Post_Science_NOOW_fixed.csv',
 'Post_Science_NOOW_fixed__internals',
 'Post_Science__internals',
 'WikiRef_220_NOOW.csv',
 '_20_Newsgroups.csv',
 '_20_Newsgroups__internals',
 '_Lenta.csv']

In [6]:
class DatasetName(Enum):
    POSTNAUKA = 'Post_Science'
    # REUTERS = 'Reuters'
    # BROWN = 'Brown'
    TWENTY_NEWSGROUPS = '20_Newsgroups'
    GOOD_RU_WIKI = 'Good_RU_Wiki'
    MKB10 = 'MKB_10'

In [7]:
DATASET_NAME_TO_DATASET_FILE_PATH = {
    DatasetName.POSTNAUKA: os.path.join(
        DATA_FOLDER_PATH, 'postnauka.csv'
    ),
    # DatasetName.REUTERS: os.path.join(
    #     DATA_FOLDER_PATH, 'Reuters.csv'
    # ),
    # DatasetName.BROWN: os.path.join(
    #     DATA_FOLDER_PATH, 'Brown.csv'
    # ),
    DatasetName.TWENTY_NEWSGROUPS: os.path.join(
        DATA_FOLDER_PATH, '20NG.csv'
    ),
    # DatasetName.AG_NEWS: os.path.join(
    #     DATA_FOLDER_PATH, 'AG_News.csv'
    # ),
    # DatasetName.WATAN: os.path.join(
    #     DATA_FOLDER_PATH, 'Watan2004.csv'
    # ),
    # DatasetName.HABRAHABR: os.path.join(
    #     DATA_FOLDER_PATH, 'Habrahabr.csv'
    # ),
    DatasetName.GOOD_RU_WIKI: os.path.join(
        DATA_FOLDER_PATH, 'ruwiki_good.txt'
    ),
    DatasetName.MKB10: os.path.join(
        DATA_FOLDER_PATH, 'MKB_10_NOOW.csv'
    ),
}

In [8]:
DATASET_NAME = DatasetName.MKB10  # select a dataset here

DATASET_FILE_PATH = DATASET_NAME_TO_DATASET_FILE_PATH[DATASET_NAME]

Checking if all OK with data, what modalities does the collection have.

In [9]:
! head -n 2 $DATASET_FILE_PATH

id,raw_text,vw_text
«Бедная_симптомами»_шизофрения,"«Бе́дная симпто́мами» шизофрени́я — подтип шизотипического расстройства в российской версии МКБ-10[1] (ранее считавшийся «простым вариантом вялопротекающей шизофрении»[2][3] и «первичным дефект-психозом»[4][3]), проявляющийся преимущественно негативными симптомами (апатией, астеническим дефектом, суженным или уплощённым аффектом, социальной аутизацией, но без бреда и галлюцинаций).


In [10]:
def get_dataset_internals_folder_path(dataset_name: DatasetName) -> str:
    return os.path.join(DATA_FOLDER_PATH, dataset_name.value + '__internals')

In [11]:
DATASET_INTERNALS_FOLDER_PATH = get_dataset_internals_folder_path(DATASET_NAME)

In [12]:
DATASET_INTERNALS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals'

In [13]:
%%time

# If using really big datasets (like Habrahabr),
# one may need to set this equal `False`
KEEP_DATASET_IN_MEMORY = True

DATASET = Dataset(
    DATASET_FILE_PATH,
    internals_folder_path=DATASET_INTERNALS_FOLDER_PATH,
    keep_in_memory=KEEP_DATASET_IN_MEMORY,
)

CPU times: user 1.71 s, sys: 190 ms, total: 1.9 s
Wall time: 1.82 s


Looking what is inside dataset's folder

In [14]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['result2', 'dict.dict', 'batches', 'vw.txt']

In [15]:
MAIN_MODALITY = '@text'

In [16]:
DATASET.get_dictionary()

artm.Dictionary(name=4aca0b61-8b11-4006-ae33-fce4bd3dd02f, num_entries=244551)

In [17]:
dictionary = DATASET.get_dictionary()

In [18]:
print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=4aca0b61-8b11-4006-ae33-fce4bd3dd02f, num_entries=244551)


In [19]:
dictionary

artm.Dictionary(name=4aca0b61-8b11-4006-ae33-fce4bd3dd02f, num_entries=46873)

In [20]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=4aca0b61-8b11-4006-ae33-fce4bd3dd02f, num_entries=22608)

In [21]:
DATASET._cached_dict = dictionary

In [22]:
DATASET.get_dictionary()

artm.Dictionary(name=4aca0b61-8b11-4006-ae33-fce4bd3dd02f, num_entries=22608)

Creating batches

In [23]:
DATASET.get_batch_vectorizer()

artm.BatchVectorizer(data_path="/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals/batches", num_batches=3)

In [24]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['result2', 'dict.dict', 'batches', 'vw.txt']

In [25]:
if KEEP_DATASET_IN_MEMORY:
    DOCUMENTS = list(DATASET._data.index)
else:
    DOCUMENTS = list(DATASET._data_index)

NUM_DOCUMENTS = len(DOCUMENTS)

print(f'Num documents: {NUM_DOCUMENTS}')

Num documents: 2036


Let's look at some text samples

In [26]:
DATASET._data.head()

,id,raw_text,vw_text
id,,,
«Бедная_симптомами»_шизофрения,«Бедная_симптомами»_шизофрения,«Бе́дная симпто́мами» шизофрени́я — подтип шиз...,«Бедная_симптомами»_шизофрения |@text бедный с...
"46,XX/46,XY","46,XX/46,XY","46,XX/46,XY (тетрагаметный химеризм) — это раз...","46,XX/46,XY |@text <person> химеризм разновидн..."
"Синдром_48,_XXXY","Синдром_48,_XXXY","Синдром 48, XXXY — это генетическое состояние,...","Синдром_48,_XXXY |@text синдром xxxy генетичес..."
"Синдром_48,_XXYY","Синдром_48,_XXYY","Синдром 48, XXYY — это аномалия хромосом, при ...","Синдром_48,_XXYY |@text синдром xxyy аномалия ..."
"Синдром_48,_XYYY","Синдром_48,_XYYY","Синдром 48, XYYY — чрезвычайно редкая анеуплои...","Синдром_48,_XYYY |@text синдром xyyy чрезвычаи..."


In [27]:
DATASET.get_possible_modalities()

{'@letter', '@ngram', '@text'}

In [28]:
MAIN_MODALITY = '@text'

In [29]:
DATASET.get_dictionary()

artm.Dictionary(name=4aca0b61-8b11-4006-ae33-fce4bd3dd02f, num_entries=22608)

In [30]:
dictionary = DATASET.get_dictionary()

In [23]:
print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=1e4e5328-9ae6-43a9-9147-27a7003c3c5d, num_entries=244551)


In [24]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=1e4e5328-9ae6-43a9-9147-27a7003c3c5d, num_entries=22608)

In [25]:
DATASET._cached_dict = dictionary

In [31]:
DATASET.get_dictionary()

artm.Dictionary(name=4aca0b61-8b11-4006-ae33-fce4bd3dd02f, num_entries=22608)

In [32]:
import scipy

from typing import List

from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)

In [33]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices=None,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        T, W = phi.shape
        # T = len(topic_indices)
        topic_indices = list(range(T))

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            # print(top, phi.shape, doc_co_occurrences.shape)
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [34]:
%%time

occurences, co_occurences = calc_doc_occurrences(DATASET, MAIN_MODALITY)

CPU times: user 4.3 s, sys: 627 ms, total: 4.93 s
Wall time: 4.88 s


In [35]:
co_occurences.shape

(22608, 22608)

In [36]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    DATASET.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [37]:
import copy


class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    @property
    def name(self):
        return self._name

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

    def compute(
            self,
            model,
            topics: List[str] = None,
            documents: List[str] = None) -> Dict[str, float]:

        values = self.call_by_topic(model)

        phi = model.get_phi()

        if topics is None:
            topics = list(phi.columns)

            if hasattr(model, 'has_bcg'):
                print(f'Detected bcg topics! Skipping for coherence computation (and will have {len(topics) - 1} topics).')

                topics = topics[:-1]
        else:
            assert False

        index2topic = {phi.columns.get_loc(t): t for t in topics}
        topic2index = {t: i for i, t in index2topic.items()}

        if hasattr(model, 'has_bcg'):
            assert list(index2topic.keys()) == list(values.keys())[:-1]
        else:
            assert list(index2topic.keys()) == list(values.keys())

        result = {
            t: float(values[topic2index[t]])
            for t in topics
        }

        assert len(result) == len(index2topic)

        return result

    def _attach(self, model: TopicModel):
        if self._name in model.custom_scores:
            print(
                f'Score with such name "{self._name}" already attached to model!'
                f' So rewriting it...'
                f' All model\'s custom scores: {list(model.custom_scores.keys())}'
            )

        # TODO: TopicModel should provide ability to add custom scores
        model.custom_scores[self.name] = copy.deepcopy(self)

## Experiment<a id="experiment"></a>

Finally we are getting to the main part!)

### Scores (for Topics and Models)<a id="scores"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we define a lot of scores (which mainly differ in initial parameters).

In [38]:
ONE_MODEL_NUM_TOPICS = 20
NUM_TOP_WORDS = 20

In [39]:
top = NUM_TOP_WORDS
target_topic_indices = list(range(ONE_MODEL_NUM_TOPICS))

coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        DATASET.get_dataset().shape[0], [top],
        # topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

intra_coherence_score = IntratextCoherenceScore(
    name='toplen_ptw',
    data=DATASET,
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
    should_compute=False,
)

diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

Other coherence score variations

And a pair of default ARTM scores (these ones are fast)

In [40]:
other_scores = [
    PerplexityScore(
        name='perplexity'
    ),
]

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [41]:
NUM_ITERATIONS = 20

In [42]:
seed = 0

In [39]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = default_train_func  # default train func

In [40]:
DATASET_INTERNALS_FOLDER_PATH

'./MKB_10__internals'

In [41]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [42]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls -alh $DATASET_INTERNALS_FOLDER_PATH

./MKB_10__internals
total 50M
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 26 18:06 .
drwxrwxr-x 10 alekseev_v mil_lab 4,0K мар 26 18:06 ..
drwxrwxr-x  2 alekseev_v mil_lab 4,0K мар 26 18:06 batches
-rw-rw-r--  1 alekseev_v mil_lab  14M мар 26 18:06 dict.dict
-rw-rw-r--  1 alekseev_v mil_lab  36M мар 26 18:06 vw.txt


In [43]:
SEARCH_RESULTS_FOLDER_PATH

'./MKB_10__internals/result'

In [44]:
! ls $SEARCH_RESULTS_FOLDER_PATH

ls: cannot access './MKB_10__internals/result': No such file or directory


In [45]:
BANK_FOLDER_PATH

'./MKB_10__internals/result/bank__0'

In [46]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [47]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [48]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 90,

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

In [49]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [50]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls $DATASET_INTERNALS_FOLDER_PATH

./MKB_10__internals
batches  dict.dict  result  vw.txt


In [51]:
optimizer._save_file_path

'./MKB_10__internals/result/search_result__0.json'

In [52]:
optimizer._topic_bank._path

'./MKB_10__internals/result/bank__0'

Fulfilling the search (get ready for a really long process!):

In [53]:
%%time

optimizer.search_for_optimum(DATASET)

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.68it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 21241.265625, 'coherence_20': 0.8081201365408722, 'diversity_euclidean': 0.05506040320149122, 'diversity_jensenshannon': 0.6354163678904116, 'diversity_hellinger': 0.7312635303582865, 'diversity_cosine': 0.7721177167251074, 'perplexity': 21241.265625, 'ppl_fair': 21241.265625, 'ppl_cheatty': 4079.3203125}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.83it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 21241.265625, 'coherence_20': 0.8081201365408722, 'diversity_euclidean': 0.05506040320149122, 'diversity_jensenshannon': 0.6354163678903747, 'diversity_hellinger': 0.7312635303582171, 'diversity_cosine': 0.7721177167251073, 'perplexity': 21241.265625, 'ppl_fair': 21241.265625, 'ppl_cheatty': 4079.3203125}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.49it/s]


...

Bank scores: {'perplexity_score': 6776.9052734375, 'coherence_20': 0.8462748392332191, 'diversity_euclidean': 0.056502865517326945, 'diversity_jensenshannon': 0.6213988557023823, 'diversity_hellinger': 0.7187754072054094, 'diversity_cosine': 0.7587374014533397, 'perplexity': 6776.9052734375, 'ppl_fair': 6776.9052734375, 'ppl_cheatty': 3492.67138671875}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.19it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 6776.9052734375, 'coherence_20': 0.8462748392332191, 'diversity_euclidean': 0.05650286551732751, 'diversity_jensenshannon': 0.6213988557024607, 'diversity_hellinger': 0.7187754072059701, 'diversity_cosine': 0.7587374014533104, 'perplexity': 6776.9052734375, 'ppl_fair': 6776.9052734375, 'ppl_cheatty': 3492.67138671875}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.23it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 6776.9052734375, 'coherence_20': 0.8462748392332191, 'diversity_euclidean': 0.05650286551732751, 'diversity_jensenshannon': 0.6213988557024641, 'diversity_hellinger': 0.7187754072059787, 'diversity_cosine': 0.7587374014533104, 'perplexity': 6776.9052734375, 'ppl_fair': 6776.9052734375, 'ppl_cheatty': 3492.67138671875}
100%|███████████████████████████████████████████████| 20/20 [14:47<00:00, 44.38s/it]
CPU times: user 22min 52s, sys: 44.7 s, total: 23min 37s
Wall time: 14min 47s


What topics we have in bank

In [54]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1       topic_2  topic_3  topic_4  topic_5
@text cdk                0.0      0.0  0.000000e+00      0.0      0.0      0.0
      внутриядерный      0.0      0.0  0.000000e+00      0.0      0.0      0.0
      соотнесение        0.0      0.0  0.000000e+00      0.0      0.0      0.0
      ангиоматозный      0.0      0.0  0.000000e+00      0.0      0.0      0.0
      хориоменингит      0.0      0.0  2.496385e-10      0.0      0.0      0.0

In [55]:
bank_topics = optimizer._topic_bank.view_topics()

In [56]:
bank_topics.shape

(22608, 6)

In [57]:
bank_topics.head()

topic_0  topic_1       topic_2  topic_3  topic_4  topic_5
@text cdk                0.0      0.0  0.000000e+00      0.0      0.0      0.0
      внутриядерный      0.0      0.0  0.000000e+00      0.0      0.0      0.0
      соотнесение        0.0      0.0  0.000000e+00      0.0      0.0      0.0
      ангиоматозный      0.0      0.0  0.000000e+00      0.0      0.0      0.0
      хориоменингит      0.0      0.0  2.496385e-10      0.0      0.0      0.0

In [60]:
bank_topics['topic_5'].sort_values(ascending=False)[:20]

@text  сустав           0.023555
       перелом          0.014489
       позвоночник      0.007490
       мышца            0.006477
       повреждение      0.006326
       артрит           0.006243
       травма           0.005468
       бедро            0.005148
       метод            0.004821
       нагрузка         0.004438
       головка          0.004422
       вывих            0.004291
       ткань            0.004209
       боль             0.004062
       нарушение        0.003741
       движение         0.003654
       тазобедренный    0.003606
       деформация       0.003596
       позвонок         0.003568
       положение        0.003560
Name: topic_5, dtype: float64

And topic scores

In [59]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5
kernel_size,3076.000000,3056.000000,2987.000000,2784.000000,2726.000000,2846.000000
coherence_20,0.822205,0.794035,0.879524,0.947137,0.754336,0.880411
distance_to_nearest,0.000000,0.839263,0.761336,0.822013,0.762815,0.764102


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [69]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [61]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [62]:
optimizer._result['num_bank_topics']

[2, 2, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]

In [63]:
len(optimizer._result['bank_topic_scores'])

20

In [64]:
optimizer._result

{'optimum': 6,
 'optimum_std': 0.0,
 'bank_scores': [{'perplexity_score': 21241.265625,
   'coherence_20': 0.8081201365408722,
   'diversity_euclidean': 0.05506040320149122,
   'diversity_jensenshannon': 0.6354163678904116,
   'diversity_hellinger': 0.7312635303582865,
   'diversity_cosine': 0.7721177167251074,
   'perplexity': 21241.265625,
   'ppl_fair': 21241.265625,
   'ppl_cheatty': 4079.3203125},
  {'perplexity_score': 21241.265625,
   'coherence_20': 0.8081201365408722,
   'diversity_euclidean': 0.05506040320149122,
   'diversity_jensenshannon': 0.6354163678903747,
   'diversity_hellinger': 0.7312635303582171,
   'diversity_cosine': 0.7721177167251073,
   'perplexity': 21241.265625,
   'ppl_fair': 21241.265625,
   'ppl_cheatty': 4079.3203125},
  {'perplexity_score': 14910.55859375,
   'coherence_20': 0.8319214169868356,
   'diversity_euclidean': 0.050882752798304065,
   'diversity_jensenshannon': 0.5908506303660231,
   'diversity_hellinger': 0.6754768331753311,
   'diversity_cos

In [65]:
optimizer._result['bank_topic_scores'][0]

[{'kernel_size': 3076,
  'coherence_20': 0.8222052116329212,
  'distance_to_nearest': 0.0},
 {'kernel_size': 3056,
  'coherence_20': 0.7940350614488233,
  'distance_to_nearest': 0.8392634795276617}]

In [66]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 3208, 'coherence_20': 0.42962387858537326},
  {'kernel_size': 2656, 'coherence_20': 0.7701415600860497},
  {'kernel_size': 3039, 'coherence_20': 0.40759208792425244},
  {'kernel_size': 2954, 'coherence_20': 0.649953932720543},
  {'kernel_size': 2633, 'coherence_20': 0.49802769849553535},
  {'kernel_size': 2987, 'coherence_20': 0.4887999883090705},
  {'kernel_size': 3076,
   'coherence_20': 0.8222052116329212,
   'distance_to_nearest': 0.0},
  {'kernel_size': 3598, 'coherence_20': 0.5338939890004764},
  {'kernel_size': 2839, 'coherence_20': 0.7126840568737308},
  {'kernel_size': 2967, 'coherence_20': 0.5541666824798857},
  {'kernel_size': 3257, 'coherence_20': 0.6344829013287363},
  {'kernel_size': 2618, 'coherence_20': 0.4364507624526562},
  {'kernel_size': 2618, 'coherence_20': 0.7874127543914287},
  {'kernel_size': 2981, 'coherence_20': 0.6213442459066951},
  {'kernel_size': 2569, 'coherence_20': 0.37798571771838246},
  {'kernel_size': 2926, 'coherence_20': 0.485001

In [67]:
optimizer._result['model_scores'][0]

{'perplexity_score': 2183.57958984375,
 'coherence_20': 0.5553686071210906,
 'diversity_euclidean': 0.04994020510914676,
 'diversity_jensenshannon': 0.5937372211329287,
 'diversity_hellinger': 0.6835495593998556,
 'diversity_cosine': 0.6570792337263397,
 'perplexity': 2183.57958984375}

In [68]:
optimizer._result['model_scores'][1]

{'perplexity_score': 2182.065185546875,
 'coherence_20': 0.5680720371310215,
 'diversity_euclidean': 0.05442068104726088,
 'diversity_jensenshannon': 0.5973725631226747,
 'diversity_hellinger': 0.6877445417877905,
 'diversity_cosine': 0.6964043550767838,
 'perplexity': 2182.065185546875}

In [69]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 6

0.8462748392332192

In [70]:
len(optimizer._result['bank_scores'])

20

In [71]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 6776.9052734375,
 'coherence_20': 0.8462748392332191,
 'diversity_euclidean': 0.05650286551732751,
 'diversity_jensenshannon': 0.6213988557024641,
 'diversity_hellinger': 0.7187754072059787,
 'diversity_cosine': 0.7587374014533104,
 'perplexity': 6776.9052734375,
 'ppl_fair': 6776.9052734375,
 'ppl_cheatty': 3492.67138671875}

In [264]:
optimizer._result['bank_topic_scores'][-1]

[{'kernel_size': 6290,
  'coherence_20': 1.9365857260048487,
  'distance_to_nearest': 0.0},
 {'kernel_size': 5987,
  'coherence_20': 2.350993531411413,
  'distance_to_nearest': 0.8670798186186544},
 {'kernel_size': 4904,
  'coherence_20': 1.665684284462788,
  'distance_to_nearest': 0.9068645569155264},
 {'kernel_size': 6419,
  'coherence_20': 3.2077123047347023,
  'distance_to_nearest': 0.9163295596263876}]

In [43]:
import artm
from topnum.model_constructor import KnownModel, init_plsa
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    transform_regularizer,
)
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )
    model.has_bcg = True  # TODO: only if init_bcg_sparse_model

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [44]:
def artm_train_func(
        dataset: Dataset,
        model_number: int,
        num_topics: int,
        num_fit_iterations: int,
        scores: List = None,
        **kwargs) -> TopicModel:
    """

    Additional Parameters
    ---------------------
    kwargs
        Some params for `_get_topic_model`, such as `cache_theta` and `num_processors`
    """

    topic_model = init_model_from_family(
        family='ARTM',
        dataset=DATASET,
        main_modality=MAIN_MODALITY,
        num_topics=ONE_MODEL_NUM_TOPICS,
        seed=model_number,
        model_params={
            'decorrelation_tau': 0.01,  # best values
            'smooth_bcg_tau': 0.05,
            'sparse_sp_tau': -0.05,
        }
    )

    num_fit_iterations_with_scores = 1

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=max(0, num_fit_iterations - num_fit_iterations_with_scores)
    )
    _fit_model_with_scores(
        topic_model,
        DATASET,
        scores,
        num_fit_iterations=num_fit_iterations_with_scores
    )

    return topic_model


def _fit_model_with_scores(
        topic_model: TopicModel,
        dataset: Dataset,
        scores: List = None,
        num_fit_iterations: int = 1):

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=num_fit_iterations
    )

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [45]:
NUM_ITERATIONS = 20

In [46]:
seed = 0

In [47]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = artm_train_func  # default train func

In [48]:
DATASET_INTERNALS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals'

In [52]:
! ls /data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals

batches  dict.dict  _result2  vw.txt


In [53]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result2'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [105]:
SEARCH_RESULTS_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals/result2'

In [106]:
! ls $SEARCH_RESULTS_FOLDER_PATH

bank__0  search_result__0.json


In [107]:
import json

In [108]:
d = json.loads(open(SEARCH_RESULTS_FOLDER_PATH + '/search_result__0.json').read())

In [110]:
d.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [112]:
len(d['bank_scores']), len(d['bank_topic_scores']), len(d['model_scores']), len(d['model_topic_scores']), len(d['num_bank_topics']), len(d['num_model_topics'])

(17, 17, 18, 17, 17, 18)

In [113]:
del d['model_scores'][-1]
del d['num_model_topics'][-1]

In [114]:
len(d['bank_scores']), len(d['bank_topic_scores']), len(d['model_scores']), len(d['model_topic_scores']), len(d['num_bank_topics']), len(d['num_model_topics'])

(17, 17, 17, 17, 17, 17)

In [115]:
with open(SEARCH_RESULTS_FOLDER_PATH + '/search_result__0.json', 'w') as f:
    f.write(json.dumps(d))

In [55]:
BANK_FOLDER_PATH

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals/result2/bank__0'

In [56]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [57]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [116]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = intra_coherence_score,  # coherence_score,
    other_topic_scores = [coherence_score],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
    documents          = DOCUMENTS,

    start_model_number   = 17,  # CHANGE HERE
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)
    #                                  70%                   80%
    topic_score_threshold_percentile = 2.344754594835432,  # 2.5358203425535235,  # DIFF ALSO HERE

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:218: UserWarning: topic_score_threshold_percentile 2.344754594835432 is less than one! It is expected to be in [0, 100]. Are you sure you want to proceed (yes/no)?
  warnings.warn(
/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:265: UserWarning: File "/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals/result2/search_result__0.json" already exists. Loading
  warnings.warn(f'File "{save_file_path}" already exists. Loading')


In [117]:
optimizer._result

{'optimum': None,
 'optimum_std': None,
 'bank_scores': [{'perplexity_score': 10775.2119140625,
   'coherence_20': 0.7207361192542046,
   'diversity_euclidean': 0.06975612378933582,
   'diversity_jensenshannon': 0.6691658019717824,
   'diversity_hellinger': 0.779559508735328,
   'diversity_cosine': 0.8041866929582749,
   'perplexity': 10775.2119140625,
   'ppl_fair': 10775.2119140625,
   'ppl_cheatty': 3620.466552734375},
  {'perplexity_score': 10775.2119140625,
   'coherence_20': 0.7207361192542046,
   'diversity_euclidean': 0.06975612378932515,
   'diversity_jensenshannon': 0.6691658019716192,
   'diversity_hellinger': 0.779559508733345,
   'diversity_cosine': 0.8041866929582674,
   'perplexity': 10775.2119140625,
   'ppl_fair': 10775.2119140625,
   'ppl_cheatty': 3620.466552734375},
  {'perplexity_score': 10775.2119140625,
   'coherence_20': 0.7207361192542046,
   'diversity_euclidean': 0.06975612378933582,
   'diversity_jensenshannon': 0.66916580197179,
   'diversity_hellinger': 0.

Checking file paths

In [60]:
optimizer._save_file_path

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals/result2/search_result__0.json'

In [61]:
optimizer._topic_bank._path

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals/result2/bank__0'

Fulfilling the search (get ready for a really long process!):

In [120]:
%%time

optimizer.search_for_optimum(DATASET)

  0%|                                                         | 0/3 [00:00<?, ?it/s]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

 50%|████████████████████████▌                        | 1/2 [01:07<01:07, 67.31s/it]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 2/2 [01:07<00:00, 33.80s/it]
Using absoulte threshold: 2.344754594835432.
Eliminating bcg topic before Hier. Cur |T| is 21, topics are: Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17',
       'topic_18', 'topic_19', 'background_20'],
      dtype='object').
Now |T| is 20, topics are: Index

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 2.85528058e-05]
 ...
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.00671581e-05 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.92206894e-05 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 7.3855790e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.0755329e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  2.8554350e-05 4.1873529e-05]
 ...
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 3.0066587e-05
  0.0000000e+00 7.4

...

!!! Bank Phi: [[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 2.85528058e-05]
 ...
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.00671581e-05 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.92206894e-05 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 7.3855790e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.0755329e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  2.8554350e-05 4.1873529e-05]
 ...
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 3.0066587e-05
  0.0000000e+00 7.4

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

 50%|████████████████████████▌                        | 1/2 [01:06<01:06, 66.26s/it]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 2/2 [01:06<00:00, 33.22s/it]
Using absoulte threshold: 2.344754594835432.
Eliminating bcg topic before Hier. Cur |T| is 21, topics are: Index(['topic_0', 'topic_1', 'topic_2', 'topic_3', 'topic_4', 'topic_5',
       'topic_6', 'topic_7', 'topic_8', 'topic_9', 'topic_10', 'topic_11',
       'topic_12', 'topic_13', 'topic_14', 'topic_15', 'topic_16', 'topic_17',
       'topic_18', 'topic_19', 'background_20'],
      dtype='object').
Now |T| is 20, topics are: Index

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


!!! Bank Phi: [[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 2.85528058e-05]
 ...
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.00671581e-05 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  3.92206894e-05 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]].
!!! Bank model Phi: [[0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 7.3855790e-06]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  0.0000000e+00 1.0755329e-05]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 0.0000000e+00
  2.8554350e-05 4.1873529e-05]
 ...
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 ... 3.0066587e-05
  0.0000000e+00 7.4

In [121]:
len(optimizer._result['model_scores'])

20

In [122]:
len(optimizer._result['num_model_topics'])

20

In [123]:
len(optimizer._result['model_topic_scores'])

20

In [124]:
optimizer._last_bank_phi.shape

(22608, 7)

In [76]:
optimizer._last_bank_phi['topic_5'].sort_values(ascending=False)[:10]

@text  расстроиство    0.034950
       личность        0.011746
       шизофрения      0.010484
       человек         0.009950
       психический     0.007883
       поведение       0.006962
       мкб             0.006765
       синдром         0.006481
       диагноз         0.005833
       социальный      0.005710
Name: topic_5, dtype: float64

In [77]:
optimizer._last_model_phi.shape

(22608, 21)

In [100]:
optimizer._last_model_phi['topic_20'].sort_values(ascending=False)[:10]

KeyError: 'topic_20'

In [125]:
optimizer._main_modality

'@text'

What topics we have in bank

In [126]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2   topic_3  topic_4  topic_5  \
@text tsd                0.0      0.0      0.0  0.000061      0.0      0.0   
      кратер             0.0      0.0      0.0  0.000000      0.0      0.0   
      cdk                0.0      0.0      0.0  0.000000      0.0      0.0   
      внутриядерный      0.0      0.0      0.0  0.000000      0.0      0.0   
      прибрежный         0.0      0.0      0.0  0.000000      0.0      0.0   

                      topic_6  
@text tsd            0.000000  
      кратер         0.000000  
      cdk            0.000029  
      внутриядерный  0.000000  
      прибрежный     0.000000

In [127]:
bank_topics = optimizer._topic_bank.view_topics()

In [128]:
bank_topics.shape

(22608, 7)

In [129]:
bank_topics.head()

topic_0  topic_1  topic_2   topic_3  topic_4  topic_5  \
@text tsd                0.0      0.0      0.0  0.000061      0.0      0.0   
      кратер             0.0      0.0      0.0  0.000000      0.0      0.0   
      cdk                0.0      0.0      0.0  0.000000      0.0      0.0   
      внутриядерный      0.0      0.0      0.0  0.000000      0.0      0.0   
      прибрежный         0.0      0.0      0.0  0.000000      0.0      0.0   

                      topic_6  
@text tsd            0.000000  
      кратер         0.000000  
      cdk            0.000029  
      внутриядерный  0.000000  
      прибрежный     0.000000

In [145]:
bank_topics['topic_0'].sort_values(ascending=False)[:20]

@text  пациент           0.021792
       препарат          0.020083
       метод             0.016310
       терапия           0.013258
       исследование      0.008842
       эффективность     0.008186
       эффект            0.007890
       применение        0.007401
       время             0.006002
       применяться       0.005824
       результат         0.005719
       больной           0.005655
       использоваться    0.004817
       врач              0.004669
       позволять         0.004607
       использовать      0.004455
       качество          0.004021
       болезнь           0.003965
       показать          0.003919
       необходимый       0.003914
Name: topic_0, dtype: float64

And topic scores

In [146]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6
kernel_size,2188.000000,2071.000000,2081.000000,2075.000000,2704.000000,2640.000000,2283.000000
toplen_ptw,2.584502,2.541149,2.365411,2.826544,2.964242,3.551999,2.427317
coherence_20,0.585257,0.462224,1.165069,0.438176,0.778271,0.713602,0.659723
distance_to_nearest,0.838955,0.833698,0.831989,0.570344,0.609341,0.515645,0.802774


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [65]:
optimizer._topic_bank._path

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10__internals/result/bank__0'

In [64]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [66]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [147]:
optimizer._result['num_bank_topics']

[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7]

In [68]:
len(optimizer._result['bank_topic_scores'])

20

In [69]:
optimizer._result

{'optimum': 5,
 'optimum_std': 10.0,
 'bank_scores': [{'perplexity_score': 26842.748046875,
   'coherence_20': 0.5206713280076062,
   'diversity_euclidean': 0.0635277267583937,
   'diversity_jensenshannon': 0.6512363231179322,
   'diversity_hellinger': 0.7538156293807475,
   'diversity_cosine': 0.7488781419022787,
   'perplexity': 26842.748046875,
   'ppl_fair': 26842.748046875,
   'ppl_cheatty': 4118.02490234375},
  {'perplexity_score': 12221.890625,
   'coherence_20': 0.6718641772061521,
   'diversity_euclidean': 0.06576941292291524,
   'diversity_jensenshannon': 0.667678847954314,
   'diversity_hellinger': 0.777060559787114,
   'diversity_cosine': 0.7880633030029592,
   'perplexity': 12221.890625,
   'ppl_fair': 12221.890625,
   'ppl_cheatty': 3699.527099609375},
  {'perplexity_score': 12221.890625,
   'coherence_20': 0.6718641772061521,
   'diversity_euclidean': 0.06576941292291524,
   'diversity_jensenshannon': 0.6676788479543131,
   'diversity_hellinger': 0.7770605597871602,
   '

In [103]:
optimizer._result['bank_topic_scores']

[[{'kernel_size': 2491,
   'coherence_20': 0.9428906252054199,
   'distance_to_nearest': 0.0},
  {'kernel_size': 2350,
   'coherence_20': 0.8066311856951885,
   'distance_to_nearest': 0.8726516982760463},
  {'kernel_size': 2420,
   'coherence_20': 0.9410456312491235,
   'distance_to_nearest': 0.8250411933273305},
  {'kernel_size': 2226,
   'coherence_20': 0.7419371396049255,
   'distance_to_nearest': 0.7790961962327442},
  {'kernel_size': 2477,
   'coherence_20': 0.7897800173209952,
   'distance_to_nearest': 0.8302451060089555},
  {'kernel_size': 2029,
   'coherence_20': 0.9042007921953766,
   'distance_to_nearest': 0.8149875113566727},
  {'kernel_size': 2081,
   'coherence_20': 1.1650692321395308,
   'distance_to_nearest': 0.8180900833750091}],
 [{'kernel_size': 2491,
   'coherence_20': 0.9428906252054199,
   'distance_to_nearest': 0.0},
  {'kernel_size': 2350,
   'coherence_20': 0.8066311856951885,
   'distance_to_nearest': 0.8726516982760463},
  {'kernel_size': 2420,
   'coherence_2

In [70]:
optimizer._result['model_scores'][0]

{'perplexity_score': 2469.907958984375,
 'coherence_20': 0.6578056100721843,
 'diversity_euclidean': 0.06349389886462474,
 'diversity_jensenshannon': 0.6449992875008204,
 'diversity_hellinger': 0.7516691302395535,
 'diversity_cosine': 0.756782482618852,
 'perplexity': 2469.907958984375,
 'toplen_ptw': 1.9238749136164537}

In [106]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 15

0.9176642698349756

In [107]:
len(optimizer._result['bank_scores'])

20

In [71]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 12221.890625,
 'coherence_20': 0.6718641772061521,
 'diversity_euclidean': 0.06576941292291526,
 'diversity_jensenshannon': 0.6676788479543315,
 'diversity_hellinger': 0.7770605597872213,
 'diversity_cosine': 0.7880633030029592,
 'perplexity': 12221.890625,
 'ppl_fair': 12221.890625,
 'ppl_cheatty': 3699.527099609375}

In [73]:
len(optimizer._result['model_topic_scores'])

20

In [72]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 2696,
   'toplen_ptw': 1.8130768133024986,
   'coherence_20': 0.4706941705736568},
  {'kernel_size': 2491,
   'toplen_ptw': 2.519329408771957,
   'coherence_20': 0.9428906252054199},
  {'kernel_size': 2188,
   'toplen_ptw': 2.5845020005647212,
   'coherence_20': 0.5852568483608062,
   'distance_to_nearest': 0.0},
  {'kernel_size': 2350,
   'toplen_ptw': 2.1494913590661584,
   'coherence_20': 0.8066311856951885},
  {'kernel_size': 1888,
   'toplen_ptw': 2.7193500221986606,
   'coherence_20': 0.5145335828913602,
   'distance_to_nearest': 0.8578327232484866},
  {'kernel_size': 2534,
   'toplen_ptw': 1.7107248346458916,
   'coherence_20': 0.4281253550341447},
  {'kernel_size': 2420,
   'toplen_ptw': 1.5977452848835931,
   'coherence_20': 0.9410456312491235},
  {'kernel_size': 3052,
   'toplen_ptw': 1.9238749136164537,
   'coherence_20': 0.618962659949188},
  {'kernel_size': 2405,
   'toplen_ptw': 1.541605040082251,
   'coherence_20': 0.6391091533344201},
  {'kernel_size':

In [76]:
for m in optimizer._result['model_topic_scores']:
    print(len([
        v for v in m
        if v['toplen_ptw'] >= 2.5358203425535235
    ]))

3
3
3
2
2
3
3
1
2
2
1
2
2
3
2
3
1
2
2
3


In [109]:
! echo $SEARCH_RESULTS_FOLDER_PATH
! ls $SEARCH_RESULTS_FOLDER_PATH

./MKB_10__internals/result2
bank__0  search_result__0.json


In [124]:
! ls Good_RU_Wiki__internals

batches  dict.dict  result  result2  vw.txt
